# Finetuning using Quantized LoRA SFT

Uncomment below lines only on first run of a session to install libraries to speed up restarts on the same session. Based on the model, context length and other parameters that you set for training, there will be a lot of trial and error and so commenting this will speed up the process a lot.

Also, make sure that whenever you want to restart on Google Colab, you should restart the session. Do not delete runtime and then restart because that will take a lot of time to setup the environment again as well as needing the following line uncommented so that the right libraries can be installed again.


In [ ]:

# %pip install torch
# %pip install -U transformers trl datasets accelerate evaluate sentencepiece bitsandbytes protobuf==3.20.3 wandb

The following 2 cells will set up the finetuning environment to properly log you in to your wandb and huggingface accounts for training logs as well as syncing with Huggingface.

In [ ]:
from google.colab import userdata
import wandb
import os

# Login to your WandB account
wandb.login(userdata.get('WANDB_API_KEY'))

# Configure project settings
os.environ["WANDB_PROJECT"] = "gemma3-270m-dolly-sft"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: xebia-shriansh (xebia-shriansh-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

The following cell chooses the model that you are trying to finetune. For this example, gemma-3-270m-it model has been used since it is a very small model which can be finetuned very quickly with the Google Colab T4 GPU instance.

Note: Make sure that you are using bitsandbytes for quantization based on your requirement. Also, based on your model, you might have to comment out or do some customization for the following line: model.config.pad_token_id = tokenizer.pad_token_id



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-3-270m-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Gets loaded in 4 bit integers
    bnb_4bit_quant_type="nf4", # Stored in this format 
    bnb_4bit_compute_dtype=torch.bfloat16, # during forward prop we use this for better precision
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="eager", # original attention that is step by step --> whereas flash attention is used
    quantization_config=bnb_config,
)
# model = model.unload()

tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Device: cuda:0
DType: torch.bfloat16


The following cell is used to choose the dataset on which you want to finetune your model. Make sure you are choosing the correct split for training. If you are training for a supervised task and not just for style adaptation, then validation and train splits will need to be created separately.

Also, make sure that the messages template that you are using is compatible with the model that you are finetuning.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:1000]")

def formatting_prompts_func(example):
    # Construct a message list for the built-in chat template
    messages = [
        {"role": "user", "content": f"{example['instruction']}\n{example['context']}"},
        {"role": "assistant", "content": example['response']}
    ]
    # Apply the template to get a single string
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

In [ ]:
print(formatting_prompts_func(dataset[0]))

<bos><start_of_turn>user
When did Virgin Australia start operating?
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<end_of_turn>
<start_of_turn>model
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<end_of_turn>






This section configures the **Supervised Fine-Tuning (SFT)** and **Low-Rank Adaptation (LoRA)** parameters. The settings below are specifically optimized for the **NVIDIA T4 GPU** (16GB VRAM) and the **Gemma 3 270M** architecture.

---

## 1. Memory & Speed Optimization (`SFTConfig`)

These arguments control the "throughput" of training - how fast the model learns without crashing the GPU.

| Argument | Setting | Purpose & Hardware Impact |
| --- | --- | --- |
| **`max_length`** | `512` | Defines the token limit. For Dolly-15k, 512 tokens cover >95% of examples while keeping memory usage low. <br> You might need to change this based on your usecase and dataset. |
| **`per_device_train_batch_size`** | `8` | The number of samples processed per step. 8 is the "sweet spot" for 270M models on a T4 to avoid OOM (Out of Memory) errors. |
| **`gradient_accumulation_steps`** | `2` | Multiplies the batch size without increasing VRAM usage. Effective Batch Size = $8 \times 2 = 16$. |
| **`learning_rate`** | `2e-4` | Learning rate can be experimented with based on multiple runs and verification of train/validation loss curve. |
| **`num_train_epochs`** | `3` | How many epochs your model will be trained for? Depending on performance requirements, you might want a higher number. <br> There is a good chance that you may not need a high number of epochs as after certain steps/epochs, the model may start to overfit. |
| **`optim`** | `"paged_adamw_8bit"` | A memory-efficient optimizer. It offloads optimizer states to CPU RAM when necessary, crucial for T4's limited memory.<br> There are other options as well which you choose based on your speed requirements and memory limitations like `"adamw_torch_fused"` |
| **`packing`** | `True` | **Major Speed Boost:** Concatenates multiple short examples into a single 512-token sequence to minimize padding waste. <br> This usually only helps if the dataset has data points where some may be smaller than others. |
| **`gradient_checkpointing`** | `True` | Saves VRAM by re-calculating the forward pass during backprop. Essential for stability on older architectures like Turing (T4). |

---

## 2. LoRA Hyperparameters (`LoraConfig`)

LoRA allows us to train only a tiny fraction (usually <1%) of the model's total parameters, making fine-tuning possible on consumer hardware.

* **`r` (Rank) = `32`**: The "size" of the adapter. Since Gemma 3 270M is small, a higher rank (32) allows it to learn more complex instructions without significant overhead. For your usecase and model, you may use a smaller rank such as 8 or 16 as well.
* **`lora_alpha` = `64`**: A scaling factor for the weights. Usually set to $2 \times r$ to maintain training stability.
* **`target_modules` = `"all-linear"`**: Automatically targets all linear layers (Attention and MLP). This ensures the model learns both factual patterns and reasoning logic.
* **`lora_dropout` = `0.05`**: A small amount of regularization to prevent the model from memorizing the Dolly dataset word-for-word.

---

## 3. Training Strategy & Use-Case

* **Model Specifics:** We are using **NF4 (4-bit)** quantization to fit the model.
* **Scheduler (`cosine`)**: Gradually lowers the learning rate following a cosine curve. This usually results in a better-performing final model compared to a `constant` rate.
* **Logging:** Every 5 steps to **Weights & Biases (WandB)** for real-time monitoring.
* **Saving:** Every `epoch`. For high-precision runs, consider switching to `save_strategy="steps"`.


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

adapter_path = "/content/gemma3-270m-dolly-checkpoints-v1"

training_args = SFTConfig(
    output_dir=adapter_path, # local Repo Name
    max_length=512,
    per_device_train_batch_size=8,    # Since GPU so batch size 8 with this model to increase speed
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",   # Use constant learning rate scheduler (cosine is better)
    num_train_epochs=3,           # How many epochs your model will be trained for? Depending on performance requirements, you might want a higher number.
    logging_steps=5,
    optim="paged_adamw_8bit",   # <— replace paged_adamw_8bit
    packing=True,
    weight_decay=0.01,
    gradient_checkpointing=True,  # Save memory in T4
    # Hugging Face Hub Checkpoint Settings
    push_to_hub=False,
    hub_strategy="all_checkpoints", # Pushes EVERY saved checkpoint to HF Hub
    save_strategy="epoch",          # Saves a checkpoint at the end of every epoch
    # save_total_limit=3,           # Optional: Keep only last 3 checkpoints on Hub to save space

    report_to="wandb",              # Uses your pre-set env vars
    run_name="dolly-epoch-comparison"
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    # modules_to_save=["lm_head", "embed_tokens"],     # This is important to set if special tokens or domain specific vocab is being trained on
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"

)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    formatting_func=formatting_prompts_func,
    peft_config=lora_config
)

In [ ]:
trainer.train()
# trainer.push_to_hub() # Final push to ensure everything is synced
trainer.save_model(adapter_path)
wandb.finish()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
5,3.493670
10,2.968713
15,2.875642
20,2.884319
25,2.659538
30,2.660002
35,2.556276
40,2.497626
45,2.700325
50,2.578509


wandb: Adding directory to artifact (/content/gemma3-270m-dolly-checkpoints-v1/checkpoint-23)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-dolly-checkpoints-v1/checkpoint-46)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-dolly-checkpoints-v1/checkpoint-69)... Done. 0.2s


train/entropy,█▇▇▇▄▃▃▂▃▃▂▁▁▁
train/epoch,▁▂▂▃▃▄▄▅▅▆▆▇██
train/global_step,▁▂▂▃▃▄▄▅▅▆▆▇██
train/grad_norm,█▄▃▃▂▁▁▁▂▁▂▁▁
train/learning_rate,██▇▇▆▅▅▄▃▂▂▁▁
train/loss,█▄▄▄▂▂▁▁▂▂▁▁▁
train/mean_token_accuracy,▁▅▅▄▆▆▇█▆▇██▇█
train/num_tokens,▁▂▂▃▃▄▄▅▅▆▆▇██
total_flos,364832193600000.0
train/entropy,2.50103
train/epoch,3


The following code snippet merges the base model with the LoRA adapter that we trained before and then saves the merged model along with the tokenizer.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_path = "/content/gemma3-270m-dolly-checkpoints-v1"                 # Choose which adapters to merge, otherwise defaults to latest
merged_model_path = "/content/gemma3-270m-dolly-v1-merged/"              # Location of merged model directory

# Load base model and tokenizer
base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Load and merge the PEFT adapters onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()                                      # Merges base model with adapter parameters that we trained before and then removes LoRA specific modules to return a standard transformer

# Save the merged model and its tokenizer
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

print(f"Model merged and saved to {merged_model_path}. Final model vocabulary size: {model.config.vocab_size}")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model merged and saved to /content/gemma3-270m-dolly-v1-merged/. Final model vocabulary size: 262144


### Test the fine-tuned model

Let's compare your fine-tuned model performance against the base model! This is a good validation for your finetuned model and showcases how its output has changed based on your dataset.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

merged_model_path = "/content/gemma3-270m-dolly-v1-merged/"

# Create Transformers inference pipeline
merged_model = AutoModelForCausalLM.from_pretrained(merged_model_path, device_map="auto")
merged_tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
pipe = pipeline("text-generation", model=merged_model, tokenizer=merged_tokenizer)
pipe_base = pipeline("text-generation", model=model_id, device_map="auto")

# Test a prompt
instruction = "What is REST API?"
context = ""
inference_messages = [
    {"role": "user", "content": f"{instruction}\n{context}"},
]

prompt = merged_tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, max_new_tokens=512)
output_base = pipe_base(prompt, max_new_tokens=512)
model_output = output[0]['generated_text'][len(prompt):].strip()
model_output_base = output_base[0]['generated_text'][len(prompt):].strip()

print(f"\nFine-tuned model output: {model_output}")

print(f"\nBase model output: {model_output_base}")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Fine-tuned model output: REST API is a technology used for communication between web servers and application servers. The term REST is the acronym for "RESTful Internet Protocol".

Base model output: REST (REST) is a programming model for building and using web applications. It's a fundamental architectural pattern for designing a web application using a client-server architecture.

Here's a breakdown of REST API concepts:

*   **Client-Server Architecture:** REST is a client-server architecture.  The client (e.g., a web browser) interacts with the server (e.g., a web application) to send data to the server.  The server then sends the data back to the client.

*   **Data (Data):**  The data that the server sends to the client.  This data is typically represented as a collection of data, often represented as a JSON-formatted data structure (e.g., a list of data items, a JSON structure, or a data structure with a `data` field).

*   **Server (The Application):** The application that the

Finally, this block pushes your model to huggingface and saves it there. This is especially important with temporary instances such as the Colab T4 instance so that you may use your model later.

In [ ]:
from huggingface_hub import ModelCard, ModelCardData, whoami

model_name = "dolly15-SFT"

username = whoami()['name']
hf_repo_id = f"{username}/{model_name}-gemma-3-270m-it"

repo_url = merged_model.push_to_hub(hf_repo_id, commit_message="Upload model")
tokenizer.push_to_hub(hf_repo_id)

card_content = f"""
---
base_model: {model_id}
tags:
- text-generation
- dolly-15k
- gemma
---
A fine-tuned model based on `{model_id}` on the dolly-15k dataset to output concise results on queries."""
card = ModelCard(card_content)
card.push_to_hub(hf_repo_id)

print(f"Uploaded to {repo_url}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...guaar8k/model.safetensors:   0%|          |  640kB /  392MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpks295bhh/tokenizer.json:  23%|##2       | 7.61MB / 33.4MB            

Uploaded to https://huggingface.co/Shriansh-Xebia/dolly15-SFT-gemma-3-270m-it/commit/9eb7f3e347426cd6114b492e9b28e0a55b6c925f
